# 18 · The numerics of control

Control computations are matrix computations, and the textbook formulas are often not the way to do them.
This notebook opens the library's algorithms: the matrix exponential by scaling and squaring with a Padé
approximant (and why the Taylor series fails), the Lyapunov equation by the real Schur form, the Riccati
equation by the Hamiltonian's invariant subspace with Newton refinement, the conditioning of polynomial
coefficients against state-space representations, and the H∞ norm by bisection on a Hamiltonian. Each is
checked against SciPy, and each has a case where the naive method breaks.

**What you will learn**
- explain scaling-and-squaring with Pade for the matrix exponential
- solve Lyapunov equations through the real Schur form
- solve Riccati equations by the Hamiltonian's invariant subspace and refine by Newton
- recognise the conditioning problems of polynomial representations and compute the H-infinity norm by bisection

**Before you start:** notebooks 03, 10 and 16  ·  **Time:** about 75 min

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import linalg as sla
from englincontrol import linalg as la, systems as sy, polynomials as pl, frequency as fr, statespace as sp, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## The matrix exponential: Taylor fails, Padé with scaling succeeds

In [2]:
def expm_taylor(A, terms=30):
    E = np.eye(len(A)); term = np.eye(len(A))
    for k in range(1, terms):
        term = term @ A / k; E = E + term
    return E
for scale in (1.0, 10.0, 30.0):
    A = scale * np.array([[-1.0, 20.0], [0.0, -2.0]])            # a non-normal matrix: large transient growth
    exact = sla.expm(A)
    print(f"|A| = {np.linalg.norm(A, 1):5.0f}: Taylor (30 terms) relative error {np.max(np.abs(expm_taylor(A) - exact)) / np.max(np.abs(exact)):.1e}, Pade 13 with scaling-and-squaring {np.max(np.abs(la.expm(A) - exact)) / np.max(np.abs(exact)):.1e}")
A = 30.0 * np.array([[-1.0, 20.0], [0.0, -2.0]])
print(f"\nfor |A| = 660 the library scales by 2^-{int(np.ceil(np.log2(np.linalg.norm(A, 1) / 5.37)))} before the Pade step and squares the result that many times")

|A| =    22: Taylor (30 terms) relative error 1.5e-15, Pade 13 with scaling-and-squaring 7.6e-16
|A| =   220: Taylor (30 terms) relative error 5.4e+10, Pade 13 with scaling-and-squaring 1.3e-14
|A| =   660: Taylor (30 terms) relative error 3.0e+33, Pade 13 with scaling-and-squaring 2.2e-14

for |A| = 660 the library scales by 2^-7 before the Pade step and squares the result that many times


The Taylor series of e^{A} converges for every A, in exact arithmetic; in floating point its terms grow to
10²⁰ before cancelling to a result of size 1, and the rounding of those terms destroys the answer. Higham's
algorithm scales A by a power of two until its norm is below 5.37, applies the [13/13] Padé approximant (exact
to double precision at that norm), and squares the result back - the method SciPy uses, and the library
reproduces it to the last digit.

## Lyapunov by the Schur form

In [3]:
rng = np.random.default_rng(0)
A = rng.normal(size=(12, 12)) - 4 * np.eye(12)
T, U = la.schur(A)
print(f"real Schur form: |U T U^T - A| = {np.max(np.abs(U @ T @ U.T - A)):.1e}, |U^T U - I| = {np.max(np.abs(U.T @ U - np.eye(12))):.1e}; T has {int(np.sum(np.abs(np.diag(T, -1)) > 1e-12))} 2 x 2 blocks (complex pairs)")
Q = np.eye(12)
P = la.lyapunov(A, Q)
print(f"Lyapunov A P + P A' + Q = 0: residual {np.linalg.norm(A @ P + P @ A.T + Q):.1e}; vs SciPy {np.max(np.abs(P - sla.solve_continuous_lyapunov(A, -Q))):.1e}")
n = 12
K = np.kron(np.eye(n), A) + np.kron(A, np.eye(n))
print(f"the Kronecker alternative solves a {n * n} x {n * n} system ({K.size * 8 / 1e6:.1f} MB, O(n^6) flops); Bartels-Stewart on the Schur form costs O(n^3)")

real Schur form: |U T U^T - A| = 1.1e-14, |U^T U - I| = 2.2e-15; T has 5 2 x 2 blocks (complex pairs)
Lyapunov A P + P A' + Q = 0: residual 2.2e-14; vs SciPy 8.2e-15
the Kronecker alternative solves a 144 x 144 system (0.2 MB, O(n^6) flops); Bartels-Stewart on the Schur form costs O(n^3)


The Lyapunov equation is linear in P and can be written as an n²×n² system through the Kronecker product -
fine for n = 4, hopeless for n = 100. Bartels and Stewart reduce A to real Schur form (quasi-triangular, by
the QR algorithm) and then solve for P block by block by back-substitution, in O(n³). The library's Schur
decomposition is a shifted QR iteration with deflation written in NumPy; it agrees with LAPACK's to rounding.

## Riccati: the Hamiltonian's invariant subspace, then Newton

In [4]:
cp = models.cart_pendulum()["system"]
A, B = cp.A, cp.B
Q, R = np.diag([10.0, 1.0, 100.0, 1.0]), np.array([[0.1]])
S = B @ np.linalg.solve(R, B.T)
H = np.block([[A, -S], [-Q, -A.T]])
lam, V = np.linalg.eig(H)
print("Hamiltonian eigenvalues come in +- pairs:", np.round(np.sort_complex(lam), 3))
stable = V[:, lam.real < 0]
P_sub = np.real(np.linalg.solve(stable[:4].T, stable[4:].T).T)
resid = lambda P_: np.linalg.norm(A.T @ P_ + P_ @ A - P_ @ S @ P_ + Q)   # noqa: E731
P_lib = la.care(A, B, Q, R)
print(f"P from the stable subspace alone: residual {resid(P_sub):.1e}; after Newton-Kleinman refinement (library): {resid(P_lib):.1e}; vs SciPy: {np.max(np.abs(P_lib - sla.solve_continuous_are(A, B, Q, R))):.1e}")
Ad, Bd = np.array([[1.0, 0.1], [0.0, 1.0]]), np.array([[0.005], [0.1]])
Pd = la.dare(Ad, Bd, np.eye(2), np.array([[0.1]]))
print(f"discrete Riccati by the structure-preserving doubling algorithm: vs SciPy {np.max(np.abs(Pd - sla.solve_discrete_are(Ad, Bd, np.eye(2), np.array([[0.1]])))):.1e}")

Hamiltonian eigenvalues come in +- pairs: [-12.298-0.947j -12.298+0.947j  -1.352-1.05j   -1.352+1.05j
   1.352-1.05j    1.352+1.05j   12.298-0.947j  12.298+0.947j]
P from the stable subspace alone: residual 3.4e-12; after Newton-Kleinman refinement (library): 4.0e-12; vs SciPy: 9.2e-14
discrete Riccati by the structure-preserving doubling algorithm: vs SciPy 4.3e-14


The Hamiltonian matrix of the LQR problem has eigenvalues in ± pairs; the stabilising Riccati solution is
P = X₂X₁⁻¹ where [X₁; X₂] spans the stable half of its invariant subspace. Computing that subspace from
eigenvectors is accurate to the conditioning of the eigenvectors (here already 10⁻¹² in the residual, so the
Newton steps - each a Lyapunov equation - have little left to polish; for badly separated eigenvalues they
gain several digits, as the exercises show). (Production codes use the Schur form of
the Hamiltonian instead of eigenvectors, for the same reason as the Lyapunov solver.) The discrete equation is
solved by the doubling algorithm, which needs no inverse of A.

## Conditioning: coefficients versus states

In [5]:
rows = []
for n in (6, 10, 14, 18):
    poles = -np.arange(1, n + 1, dtype=float)
    c = pl.from_roots(poles)
    roots_from_coeffs = np.sort(pl.roots(c).real)
    G = sy.zpk([], poles, 1.0)
    bal, _ = sy.balanced_realisation(G.to_ss())
    rows.append((n, c.max(), np.max(np.abs(roots_from_coeffs - np.sort(poles))), np.max(np.abs(np.sort(bal.poles().real) - np.sort(poles))), np.max(np.abs(np.sort(G.to_ss().poles().real) - np.sort(poles)))))
print(pd.DataFrame(rows, columns=["order", "largest coefficient", "pole error from coefficients", "from the balanced ss", "from the companion ss"]).to_string(index=False, float_format="%.1e"))

 order  largest coefficient  pole error from coefficients  from the balanced ss  from the companion ss
     6              1.8e+03                       5.5e-13               3.9e-11                5.5e-13
    10              1.3e+07                       2.8e-09               2.8e-06                2.8e-09
    14              3.9e+11                       6.3e-07               6.3e-01                6.3e-07
    18              3.4e+16                       4.4e-03               1.2e+01                4.4e-03


Roots at −1, …, −18 give coefficients up to 10¹⁶; recovering the roots from them (the companion matrix's
eigenvalues - which is also what the companion-form state-space model carries) loses digits steadily, to
10⁻³ at order 18, and the balanced realisation computed from it through Gramians loses far more (its
transformation is ill-conditioned, as notebook 16 found). The accurate representation of such a system is
the modal one, built directly from the known poles, never from the coefficients. The practical rules: never
form a high-order transfer function as a coefficient list if the poles are known; build state-space models
from physics or from modal data; and when a transformation is needed, check its condition number before
trusting what comes out of it.

## The H-infinity norm by bisection

In [6]:
G = models.quarter_car()["system"]
norm, w_pk = fr.hinf_norm(G)
w = np.logspace(-1, 3, 50000)
sweep = np.max([np.linalg.norm(fr.freqresp(G, wk), 2) for wk in w[::10]])
print(f"H-inf norm of the quarter car (2 x 2) by Boyd-Balakrishnan bisection: {norm:.4f} (peak near {w_pk:.1f} rad/s); a 5000-point sweep gives {sweep:.4f}")

H-inf norm of the quarter car (2 x 2) by Boyd-Balakrishnan bisection: 508.0996 (peak near 63.7 rad/s); a 5000-point sweep gives 508.0948


The peak gain over frequency is found without a sweep: γ exceeds ‖G‖∞ exactly when a Hamiltonian matrix built
from (A, B, C, D, γ) has no eigenvalue on the imaginary axis, so a bisection on γ with one eigenvalue
computation per step converges to the norm to any precision - the engine inside H∞ synthesis. A sweep can
miss a sharp peak between grid points; the bisection cannot.

## Inside the algorithm: one Newton-Kleinman step is a Lyapunov equation

In [7]:
Ak = A - S @ P_sub
dP = la.sylvester(Ak.T, Ak, -(A.T @ P_sub + P_sub @ A - P_sub @ S @ P_sub + Q))
print(f"after one Newton step from the subspace solution: residual {resid(P_sub + dP):.1e} (was {resid(P_sub):.1e})")

after one Newton step from the subspace solution: residual 4.4e-13 (was 3.4e-12)


## Exercises
1. Time `linalg.expm`, the Taylor series and `scipy.linalg.expm` for random 10 x 10, 50 x 50 and 200 x 200 matrices of norm 1 and 100; where does the Taylor series lose accuracy, and how does the cost of the library's method scale?
2. Compute the Riccati solution for the cart-pendulum with R from 1 to 1e-8: at what point does the subspace-only solution (before Newton) lose accuracy, and does Newton always recover it? Explain with the Hamiltonian's eigenvalue separation.
3. **Implement it yourself:** write the bisection for the H-infinity norm of a SISO system (gamma is an upper bound iff the Hamiltonian has no imaginary eigenvalues) and reproduce `frequency.hinf_norm` for the prototype second-order system and for a lightly damped one with zeta = 0.01.